In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.v2 as v2
import os
import cv2
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import scipy.io as io
! wget https://zenodo.org/record/4126613/files/CALTECH.zip -O caltech.zip
! unzip -q caltech.zip
annot_path = "/content/CALTECH/CALTECH_Annotations/butterfly/annotation_0001.mat"
import scipy.io as io

f = io.loadmat(annot_path)
f
f['box_coord']
img_path = "/content/CALTECH/CALTECH_Dataset/butterfly/image_0001.jpg"
import cv2

img = cv2.imread(img_path)
img
y_min, y_max, x_min, x_max = f['box_coord'][0]
new_img = cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
new_img
root_img = "/content/CALTECH/CALTECH_Dataset/"
root_annot = "/content/CALTECH/CALTECH_Annotations/"
import os

os.listdir(root_img)
folders = os.listdir(root_img)
folders[0]
os.path.join(root_img, folders[0])
class_path = os.path.join(root_img, folders[0])
os.listdir(class_path)
root_img = "/content/CALTECH/CALTECH_Dataset/"
root_annot = "/content/CALTECH/CALTECH_Annotations/"

imgs_path = []
for class_name in os.listdir(root_img):
    class_path = os.path.join(root_img, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        imgs_path.append(img_path)
imgs_path[124]
root_img = "/content/CALTECH/CALTECH_Dataset/"
root_annot = "/content/CALTECH/CALTECH_Annotations/"

imgs_path = []
for class_name in os.listdir(root_img):
    class_path = os.path.join(root_img, class_name)
    class_annot_path = os.path.join(root_annot, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        imgs_path.append(img_path)
class_annot_path = os.path.join(root_annot, class_name)
class_annot_path
img_name
img_name.split('.')
img_name.replace(".jpg", ".mat")
img_name.replace(".jpg", ".mat").replace("image", "annotation")
root_img = "/content/CALTECH/CALTECH_Dataset/"
root_annot = "/content/CALTECH/CALTECH_Annotations/"

imgs_path = []
annots_path = []
for class_name in os.listdir(root_img):
    class_path = os.path.join(root_img, class_name)
    class_annot_path = os.path.join(root_annot, class_name)
    for img_name in os.listdir(class_path):
        annot_name = img_name.replace(".jpg", ".mat").replace("image", "annotation")
        img_path = os.path.join(class_path, img_name)
        annot_path = os.path.join(class_annot_path, annot_name)
        imgs_path.append(img_path)
        annots_path.append(annot_path)
idx = 72
imgs_path[idx], annots_path[idx]
idx = 173
img = cv2.imread(imgs_path[idx])
mat = io.loadmat(annots_path[idx])

y_min, y_max, x_min, x_max = mat['box_coord'][0]
new_img = cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
new_img
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


class Caltech(Dataset):
    def __init__(self, root_img, root_annot, transform=None):
        self.root_img = root_img
        self.root_annot = root_annot
        self.transform = transform

        self.imgs_path = []
        self.annots_path = []
        self.labels = []

        for class_idx, class_name in enumerate(os.listdir(root_img)):
            class_path = os.path.join(root_img, class_name)
            class_annot_path = os.path.join(root_annot, class_name)
            for img_name in os.listdir(class_path):
                annot_name = img_name.replace(".jpg", ".mat").replace("image", "annotation")
                img_path = os.path.join(class_path, img_name)
                annot_path = os.path.join(class_annot_path, annot_name)
                self.imgs_path.append(img_path)
                self.annots_path.append(annot_path)
                self.labels.append(class_idx)

    def __len__(self):
        return len(imgs_path)

    def __getitem__(self, idx):
        img = cv2.imread(self.imgs_path[idx])
        mat = io.loadmat(self.annots_path[idx])
        label = self.labels[idx]

        y_min, y_max, x_min, x_max = mat['box_coord'][0]

        annot = torch.tensor([x_min, y_min, x_max, y_max]).float()

        if self.transform:
            img = self.transform(img)

        return img, annot, label


import torch
import torchvision.transforms.v2 as v2

root_img = "/content/CALTECH/CALTECH_Dataset/"
root_annot = "/content/CALTECH/CALTECH_Annotations/"

transform = v2.Compose([
    v2.ToPILImage(),
    v2.Resize(256),
    v2.CenterCrop(224),
    v2.ToTensor()
])

dataset = Caltech(root_img, root_annot, transform)
loader = DataLoader(dataset, batch_size=32, shuffle=True)
for img, annot, label in loader:
    break
import matplotlib.pyplot as plt

idx = 14
x_min, y_min, x_max, y_max = annot[idx]
img_ = cv2.cvtColor(img[idx].permute(1, 2, 0).numpy(), cv2.COLOR_BGR2RGB)
new_img = cv2.rectangle(img_, (int(x_min), int(y_min)), (int(x_max), int(y_max)), (0, 255, 0), 2)
plt.imshow(new_img)


class Caltech(Dataset):
    def __init__(self, root_img, root_annot, transform=None):
        self.root_img = root_img
        self.root_annot = root_annot
        self.transform = transform

        self.imgs_path = []
        self.annots_path = []
        self.labels = []

        for class_idx, class_name in enumerate(os.listdir(root_img)):
            class_path = os.path.join(root_img, class_name)
            class_annot_path = os.path.join(root_annot, class_name)
            for img_name in os.listdir(class_path):
                annot_name = img_name.replace(".jpg", ".mat").replace("image", "annotation")
                img_path = os.path.join(class_path, img_name)
                annot_path = os.path.join(class_annot_path, annot_name)
                self.imgs_path.append(img_path)
                self.annots_path.append(annot_path)
                self.labels.append(class_idx)

    def __len__(self):
        return len(imgs_path)

    def __getitem__(self, idx):
        img = cv2.imread(self.imgs_path[idx])
        mat = io.loadmat(self.annots_path[idx])
        label = self.labels[idx]

        h, w, c = img.shape
        y_min, y_max, x_min, x_max = mat['box_coord'][0]

        annot = torch.tensor([x_min / w, y_min / h, x_max / w, y_max / h]).float()

        if self.transform:
            img = self.transform(img)

        return img, annot, label


root_img = "/content/CALTECH/CALTECH_Dataset/"
root_annot = "/content/CALTECH/CALTECH_Annotations/"

transform = v2.Compose([
    v2.ToPILImage(),
    v2.Resize((224, 224)),
    v2.ToTensor()
])

dataset = Caltech(root_img, root_annot, transform)
loader = DataLoader(dataset, batch_size=32, shuffle=True)
for img, annot, label in loader:
    break
idx = 12
x_min, y_min, x_max, y_max = annot[idx]
img_ = cv2.cvtColor(img[idx].permute(1, 2, 0).numpy(), cv2.COLOR_BGR2RGB)
new_img = cv2.rectangle(img_, (int(x_min * 224), int(y_min * 224)), (int(x_max * 224), int(y_max * 224)), (0, 255, 0),
                        2)
plt.imshow(new_img)

class Caltech(Dataset):
    def __init__(self, root_img, root_annot, transform=None):
        self.root_img = root_img
        self.root_annot = root_annot
        self.transform = transform

        self.imgs_path = []
        self.annots_path = []
        self.labels = []

        for class_idx, class_name in enumerate(os.listdir(root_img)):
            class_path = os.path.join(root_img, class_name)
            class_annot_path = os.path.join(root_annot, class_name)
            for img_name in os.listdir(class_path):
                annot_name = img_name.replace(".jpg", ".mat").replace("image", "annotation")
                img_path = os.path.join(class_path, img_name)
                annot_path = os.path.join(class_annot_path, annot_name)
                self.imgs_path.append(img_path)
                self.annots_path.append(annot_path)
                self.labels.append(class_idx)

    def __len__(self):
        return len(self.imgs_path)

    def __getitem__(self, idx):
        img = cv2.imread(self.imgs_path[idx])
        mat = io.loadmat(self.annots_path[idx])
        label = self.labels[idx]

        h, w, c = img.shape
        y_min, y_max, x_min, x_max = mat['box_coord'][0]

        annot = torch.tensor([x_min / w, y_min / h, x_max / w, y_max / h]).float()

        if self.transform:
            img = self.transform(img)

        return img, annot, label


root_img = "/content/CALTECH/CALTECH_Dataset/"
root_annot = "/content/CALTECH/CALTECH_Annotations/"

transform = v2.Compose([
    v2.ToPILImage(),
    v2.Resize((224, 224)),
    v2.ToTensor()
])

dataset = Caltech(root_img, root_annot, transform)
loader = DataLoader(dataset, batch_size=32, shuffle=True)
for img, annot, label in loader:
    break
# out = model(img)
#
model = torchvision.models.resnet18(weights=None)
model.fc = nn.Identity()
model


class localizer(nn.Module):
    def __init__(self):
        super(localizer, self).__init__()
        self.feat_ext = torchvision.models.resnet18(weights=None)
        in_features = self.feat_ext.fc.in_features
        # self.model.fc = nn.Linear(in_features=self.model.fc.in_features, out_features=4)
        self.feat_ext.fc = nn.Identity()

        self.loc_head = nn.Sequential(
            nn.Linear(in_features=in_features, out_features=4),
            nn.Sigmoid()
        )
        self.cls_head = nn.Sequential(
            nn.Linear(in_features=in_features, out_features=3),
        )

    def forward(self, x):
        x = self.feat_ext(x)
        bbox = self.loc_head(x)
        pred = self.cls_head(x)

        return pred, bbox


model = localizer()
preds, bboxes = model(img)
preds
bboxes
idx = 10
x_min, y_min, x_max, y_max = annot[idx]
img_ = cv2.cvtColor(img[idx].permute(1, 2, 0).numpy(), cv2.COLOR_BGR2RGB)
# img_ = cv2.rectangle(img_, (int(x_min * 224), int(y_min * 224)), (int(x_max * 224), int(y_max * 224)), (0, 255, 0), 2)
img_ = cv2.circle(img_, (int(x_min * 224), int(y_min * 224)), radius=0, color=(255, 0, 255), thickness=10)
img_ = cv2.circle(img_, (int(x_max * 224), int(y_max * 224)), radius=0, color=(255, 0, 255), thickness=10)
x_min, y_min, x_max, y_max = bboxes[idx]
# img_ = cv2.rectangle(img_, (int(x_min * 224), int(y_min * 224)), (int(x_max * 224), int(y_max * 224)), (255, 0, 0), 2)
img_ = cv2.circle(img_, (int(x_min * 224), int(y_min * 224)), radius=0, color=(0, 255, 255), thickness=10)
img_ = cv2.circle(img_, (int(x_max * 224), int(y_max * 224)), radius=0, color=(0, 255, 255), thickness=10)
plt.imshow(img_)
nn.CrossEntropyLoss()(preds, label)
nn.MSELoss()(bboxes, annot)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = localizer()
cls_criterion = nn.CrossEntropyLoss()
reg_criterion = nn.MSELoss()

model = model.to(device)
cls_criterion = cls_criterion.to(device)
reg_criterion = reg_criterion.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
from sklearn.metrics import accuracy_score


def train(loader, model, cls_cri, reg_cri, optimizer, device):
    model.train()
    metrics = {
        "reg_loss": [],
        "cls_loss": [],
        "loss": [],
        "acc": [],
    }
    for data, annot, label in loader:
        optimizer.zero_grad()

        data = data.to(device)
        label = label.to(device)
        annot = annot.to(device)

        pred, bboxes = model(data)
        cls_loss = cls_cri(pred, label)
        reg_loss = reg_cri(bboxes, annot)
        loss = cls_loss + reg_loss
        loss.backward()
        optimizer.step()

        acc = accuracy_score(label.detach().cpu(), pred.argmax(dim=1).detach().cpu())
        metrics['acc'].append(acc)
        metrics['loss'].append(loss.item())
        metrics['reg_loss'].append(reg_loss.item())
        metrics['cls_loss'].append(cls_loss.item())

    return metrics


epochs = 10
for epoch in range(epochs):
    metrics = train(loader, model, cls_criterion, reg_criterion, optimizer, device)
    print(
        f"Epoch {epoch + 1}/{epochs}: Train loss: {np.mean(metrics['loss'])}, Train acc: {np.mean(metrics['acc'])}, cls loss: {np.mean(metrics['cls_loss'])}, reg acc: {np.mean(metrics['reg_loss'])}")
